# Export Gebäude Stadt Luzern (Baujahr 1970–2000)

Dieses Notebook dokumentiert die Einzelschritte zum Download, Filtern und Exportieren der Datensätze aus:
https://daten.geo.lu.ch/browser/#/collections/KGWRPUBL_COL_V3/
Die Arbeitsstruktur ist jetzt getrennt in `notebook/` und `output/`.

## 1) Datenquelle und lokale Pfade

In [7]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import urllib.request
import zipfile

STAC_ITEM_URL = 'https://daten.geo.lu.ch/api/stac/v1.0/collections/KGWRPUBL_COL_V3/items?limit=1'
GPKG_ZIP_URL = 'https://download.geo.lu.ch/api/stac/v1.0/downloads/KGWRPUBL_COL_V3/KGWRPUBL_COL/KGWRPUBL_COL_V3_gpkg.zip'

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == 'notebook' else cwd
base = project_root / 'output' / 'kgwr'
zip_path = base / 'KGWRPUBL_COL_V3_gpkg.zip'
extract_dir = base
gpkg_path = base / 'daten' / 'KGWRGPUB_DS_V3_20260812.gpkg'
csv_path = base / 'kgwr_gebaeude_luzern_baujahr_1970_2000.csv'

base.mkdir(parents=True, exist_ok=True)
print('Basisordner:', base.resolve())

Basisordner: /Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/kgwr


## 2) GeoPackage herunterladen und entpacken

In [8]:
urllib.request.urlretrieve(GPKG_ZIP_URL, zip_path)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

print('ZIP:', zip_path.resolve())
print('GPKG vorhanden:', gpkg_path.exists())

ZIP: /Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/kgwr/KGWRPUBL_COL_V3_gpkg.zip
GPKG vorhanden: True


## 3) Daten mit pandas auf Stadt Luzern und Baujahr 1970–2000 filtern

In [9]:
gdf = gpd.read_file(gpkg_path, layer='KGWRGPUB_V3_PT')
filtered = gdf[(gdf['BFS_GEMEINDE'] == 'Luzern') & (gdf['GBAUJ'].between(1970, 2000))].copy()
filtered = filtered.sort_values('EGID')

print('Gefilterte Datensätze:', len(filtered))

/Users/padrian/miniconda3/envs/georg-moersch-bildanalyse/lib/python3.9/site-packages/pyogrio/raw.py:198: RuntimeWarning: Non-conformant content for record 111206 in column GWAERDATH1, 2024-06-04T00:00:00.0Z, successfully parsed
  return ogr_read(


Gefilterte Datensätze: 2406


## 4) Ergebnis als CSV exportieren

In [10]:
filtered.drop(columns='geometry', errors='ignore').to_csv(csv_path, index=False, encoding='utf-8')
print('CSV geschrieben:', csv_path.resolve())

CSV geschrieben: /Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/kgwr/kgwr_gebaeude_luzern_baujahr_1970_2000.csv


## 5) Plausibilitätscheck

In [11]:
df = pd.read_csv(csv_path)

print('Zeilen:', len(df))
print('Baujahr min/max:', int(df['GBAUJ'].min()), int(df['GBAUJ'].max()))
print('Gemeinden:', sorted(df['BFS_GEMEINDE'].dropna().unique())[:5])
print('Erste 5 Zeilen:')
display(df.head(5))


Zeilen: 2406
Baujahr min/max: 1970 2000
Gemeinden: ['Luzern']
Erste 5 Zeilen:


,EGID,BFS_NR,GDEHISTID,BFS_GEMEINDE,ESID,STRNAMK1_HPT,STRNAMK1,DEINR,DEINR_ZUS_HPT,GBEZ,...,GWAERDATW1,GENW2,GENW2_TXT,GWAERSW2,GWAERSW2_TXT,GWAERZW2,GWAERZW2_TXT,GWAERDATW2,KOORD_ORIGIN,DAT_VERARB
0,208349,1061.0,15600.0,Luzern,10014627.0,Bennenegg,Bennenegg,22.0,NaN,NaN,...,2024-06-04 00:00:00+00:00,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00
1,208350,1061.0,15600.0,Luzern,10014627.0,Bennenegg,Bennenegg,24.0,NaN,NaN,...,2024-06-04 00:00:00+00:00,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00
2,208351,1061.0,15600.0,Luzern,10014627.0,Bennenegg,Bennenegg,26.0,NaN,NaN,...,2024-06-04 00:00:00+00:00,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00
3,208352,1061.0,15600.0,Luzern,10014627.0,Bennenegg,Bennenegg,28.0,NaN,NaN,...,2024-06-04 00:00:00+00:00,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00
4,208353,1061.0,15600.0,Luzern,10014627.0,Bennenegg,Bennenegg,30.0,NaN,NaN,...,2018-11-08 00:00:00+00:00,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00


In [12]:
alpenquai = filtered[filtered['STRNAMK1_HPT'].astype(str).str.contains('Alpenquai', case=False, na=False)].copy()

print('Treffer:', len(alpenquai))
display(alpenquai[['STRNAMK1_HPT', 'DEINR', 'GKLAS_TXT', 'GBEZ']])

Treffer: 26


,EGID,BFS_NR,GDEHISTID,BFS_GEMEINDE,ESID,STRNAMK1_HPT,STRNAMK1,DEINR,DEINR_ZUS_HPT,GBEZ,...,GENW2,GENW2_TXT,GWAERSW2,GWAERSW2_TXT,GWAERZW2,GWAERZW2_TXT,GWAERDATW2,KOORD_ORIGIN,DAT_VERARB,geometry
27636,213892,1061.0,15600.0,Luzern,10069513.0,Alpenquai,Alpenquai,12,None,None,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666774.169 1210992.388)
27637,213893,1061.0,15600.0,Luzern,10069513.0,Alpenquai,Alpenquai,14,None,None,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666788.349 1210976.548)
27710,213992,1061.0,15600.0,Luzern,10069513.0,Alpenquai,Alpenquai,40,None,None,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2667011.179 1210761.078)
27713,213995,1061.0,15600.0,Luzern,10069513.0,Alpenquai,Alpenquai,34,None,None,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666964.529 1210805.074)
27714,213996,1061.0,15600.0,Luzern,10069513.0,Alpenquai,Alpenquai,34,None,None,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666946.904 1210794.504)
27726,214008,1061.0,15600.0,Luzern,10069513.0,Alpenquai,Alpenquai,34,None,None,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666921.835 1210849.201)
27727,214009,1061.0,15600.0,Luzern,10069513.0,Alpenquai,Alpenquai,34,None,None,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666911.074 1210831.699)
72269,214012,1061.0,15600.0,Luzern,10069513.0,Alpenquai,Alpenquai,30,None,None,...,7500.0,Keine,869.0,Gemäss Baubewilligung,7600.0,Kein Wärmeerzeuger,2023-06-30 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666886.774 1210878.372)
27732,214018,1061.0,15600.0,Luzern,10069513.0,Alpenquai,Alpenquai,33,None,Bootshaus,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2667043.574 1210802.486)
76514,2373941,1061.0,15600.0,Luzern,10069513.0,Alpenquai,Alpenquai,11,None,Bootsreparatur- Werkstätte SNG,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666796.955 1211067.937)


In der Spalte GKLAS_TXT werden die Gebäude Nutzunge zugeschrieben. Im nächsten Schritt sammeln wir alle Werte dieser Spalte, um einen Überblick zu erhalten.

In [13]:
unique_gklas_txt = sorted(filtered['GKLAS_TXT'].dropna().unique())

display(unique_gklas_txt)

['Andere Beherbergung',
 'Andere landw. Geb.',
 'Behälter, Silo, Lager',
 'Bürogebäude',
 'Garagengebäude',
 'Gebäude mit 1 Wohnung',
 'Gebäude mit 2 Wohnungen',
 'Gebäude mit 3+ Whgen',
 'Gross- und Einzelhandel',
 'Hotelgebäude',
 'Industriegebäude',
 'Kirche / Kultgebäude',
 'Krankenhaus',
 'Kultur-/Freizeitstätte',
 'Landw. Betriebsgebäude',
 'Museum / Bibliothek',
 'Pflanzenbau',
 'Schul-/Hochschulgebäude',
 'Sonstiger Hochbau',
 'Sporthalle',
 'Verkehr / Kommunikation',
 'Wohngeb.f.Gemeinschaften']

Wieviele Gebäude werden den jeweiligen Nutzunge zugeordnet?

In [19]:
gklas_counts = filtered['GKLAS_TXT'].value_counts(dropna=True)
display(gklas_counts)

GKLAS_TXT
Gebäude mit 3+ Whgen        714
Garagengebäude              629
Gebäude mit 1 Wohnung       278
Sonstiger Hochbau           163
Industriegebäude            120
Bürogebäude                  91
Behälter, Silo, Lager        70
Kultur-/Freizeitstätte       68
Gebäude mit 2 Wohnungen      67
Landw. Betriebsgebäude       48
Schul-/Hochschulgebäude      37
Verkehr / Kommunikation      25
Wohngeb.f.Gemeinschaften     22
Gross- und Einzelhandel      20
Kirche / Kultgebäude         14
Krankenhaus                  13
Sporthalle                    9
Hotelgebäude                  8
Museum / Bibliothek           5
Andere Beherbergung           3
Andere landw. Geb.            1
Pflanzenbau                   1
Name: count, dtype: int64

Nun können wir nach Nutzunge filtern

In [14]:
kirche = filtered[filtered['GKLAS_TXT'] == 'Kirche / Kultgebäude'].copy()

print('Kirche-/Kultgebäude:', len(kirche))
display(kirche)

Kirche-/Kultgebäude: 14


,EGID,BFS_NR,GDEHISTID,BFS_GEMEINDE,ESID,STRNAMK1_HPT,STRNAMK1,DEINR,DEINR_ZUS_HPT,GBEZ,...,GENW2,GENW2_TXT,GWAERSW2,GWAERSW2_TXT,GWAERZW2,GWAERZW2_TXT,GWAERDATW2,KOORD_ORIGIN,DAT_VERARB,geometry
71511,208605,1061.0,15600.0,Luzern,10085054.0,Blattenmoosstrasse,Blattenmoosstr.,8,None,None,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2663391.134 1211426.949)
26044,212019,1061.0,15600.0,Luzern,10052330.0,Spitalstrasse,Spitalstr.,91,None,None,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2664935.296 1212022.57)
26889,213013,1061.0,15600.0,Luzern,10015055.0,Winkelriedstrasse,Winkelriedstr.,5,None,Pfarreiheim Barfüesser,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2665827.524 1211288.585)
27734,214023,1061.0,15600.0,Luzern,10085846.0,Langensandstrasse,Langensandstr.,1,None,Pfarreiheim,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2667194.197 1210258.622)
75650,2060007,1061.0,15600.0,Luzern,10015319.0,Zollhausstrasse,Zollhausstr.,5,None,None,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2663914.205 1213243.864)
80038,191033063,1061.0,15600.0,Luzern,10014651.0,Eichenstrasse,Eichenstr.,23,None,Friedhofhalle,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2663747.213 1212413.364)
85606,191731542,1061.0,15600.0,Luzern,10190937.0,St.-Leodegar-Strasse,St.-Leodegar-Str.,6,None,Pfarrei-Saal St Leodegar,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666500.867 1211933.536)
86097,191744930,1061.0,15600.0,Luzern,10012229.0,Ibachstrasse,Ibachstr.,2,None,Leichenhaus,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2664618.548 1212678.87)
86615,191761917,1061.0,15600.0,Luzern,10014839.0,Landschaustrasse,Landschaustr.,6,None,Seelsorgestation ABZUBRECHEN,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666613 1212545)
86735,191765034,1061.0,15600.0,Luzern,10085057.0,Luzernerstrasse,Luzernerstr.,90,None,Kirche,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2663174.301 1211454.284)


In [16]:
museum = filtered[filtered['GKLAS_TXT'] == 'Museum / Bibliothek'].copy()

print('Museum / Bibliothek:', len(museum))
display(museum)

Museum / Bibliothek: 5


,EGID,BFS_NR,GDEHISTID,BFS_GEMEINDE,ESID,STRNAMK1_HPT,STRNAMK1,DEINR,DEINR_ZUS_HPT,GBEZ,...,GENW2,GENW2_TXT,GWAERSW2,GWAERSW2_TXT,GWAERZW2,GWAERZW2_TXT,GWAERDATW2,KOORD_ORIGIN,DAT_VERARB,geometry
79313,190648151,1061.0,15600.0,Luzern,10084480.0,Lidostrasse,Lidostr.,5,None,IMAX,...,7520.0,Gas,869.0,Gemäss Baubewilligung,7630.0,Heizkessel (generisch),2023-02-28 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2668117.736 1211800.121)
79488,190673094,1061.0,15600.0,Luzern,10084480.0,Lidostrasse,Lidostr.,5,None,Halle Luft- und Raumfahrt /,...,7520.0,Gas,869.0,Gemäss Baubewilligung,7630.0,Heizkessel (generisch),2023-02-28 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2668205.884 1211628.5)
87201,191813464,1061.0,15600.0,Luzern,10084480.0,Lidostrasse,Lidostr.,5,None,Halle Schienenverkehr 2 + 3,...,7520.0,Gas,869.0,Gemäss Baubewilligung,7630.0,Heizkessel (generisch),2023-02-28 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2668214.706 1211779.2)
87204,191813472,1061.0,15600.0,Luzern,10084480.0,Lidostrasse,Lidostr.,7,None,Hans Erni-Museum,...,7520.0,Gas,869.0,Gemäss Baubewilligung,7630.0,Heizkessel (generisch),2023-02-28 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2668242.68 1211596.685)
87205,191813473,1061.0,15600.0,Luzern,10084480.0,Lidostrasse,Lidostr.,7,None,Büroanbau Erni Museum,...,7520.0,Gas,869.0,Gemäss Baubewilligung,7630.0,Heizkessel (generisch),2023-02-28 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2668252.448 1211586.872)


In [18]:
spital = filtered[filtered['GKLAS_TXT'] == 'Krankenhaus'].copy()

print('Krankenhaus:', len(spital))
display(spital)

Krankenhaus: 13


,EGID,BFS_NR,GDEHISTID,BFS_GEMEINDE,ESID,STRNAMK1_HPT,STRNAMK1,DEINR,DEINR_ZUS_HPT,GBEZ,...,GENW2,GENW2_TXT,GWAERSW2,GWAERSW2_TXT,GWAERZW2,GWAERZW2_TXT,GWAERDATW2,KOORD_ORIGIN,DAT_VERARB,geometry
71718,210135,1061.0,15600.0,Luzern,10114667.0,Lützelmattstrasse,Lützelmattstr.,1,None,UMNUTZUNG IN BÜRO,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2667543.944 1212368.158)
71719,210136,1061.0,15600.0,Luzern,10114667.0,Lützelmattstrasse,Lützelmattstr.,3,None,None,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2667584.416 1212398.077)
71995,212127,1061.0,15600.0,Luzern,10015072.0,Kantonsspital,Kantonsspital,31,None,Bettenhochhaus,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2665212.22 1212397.418)
79306,190648111,1061.0,15600.0,Luzern,10015072.0,Kantonsspital,Kantonsspital,31,None,Verwaltungsgebäude,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2665228.382 1212339.442)
79685,190754129,1061.0,15600.0,Luzern,10015072.0,Kantonsspital,Kantonsspital,30,None,Magnetresonanz-Tomographie,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2665247.073 1212290.174)
79782,190813749,1061.0,15600.0,Luzern,10074197.0,St.-Anna-Strasse,St.-Anna-Str.,36,None,Labor- und Bürogebäude Trakt E,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2667499.142 1212286.052)
79783,190813750,1061.0,15600.0,Luzern,10074197.0,St.-Anna-Strasse,St.-Anna-Str.,36,None,Trakt E und F,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2667512.102 1212302.981)
79784,190813751,1061.0,15600.0,Luzern,10074197.0,St.-Anna-Strasse,St.-Anna-Str.,34,None,Trakt D,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2667465.601 1212287.73)
81244,191432451,1061.0,15600.0,Luzern,10015072.0,Kantonsspital,Kantonsspital,31,None,Spitalbau Breitfuss,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2665169.068 1212359.443)
83277,191680091,1061.0,15600.0,Luzern,10015072.0,Kantonsspital,Kantonsspital,28,None,Strahlentherapie/Onkologie,...,7500.0,Keine,869.0,Gemäss Baubewilligung,7600.0,Kein Wärmeerzeuger,2017-12-28 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2665133.087 1212311.526)


In [21]:
gemeinschaft = filtered[filtered['GKLAS_TXT'] == 'Wohngeb.f.Gemeinschaften'].copy()

print('Wohngeb.f.Gemeinschaften:', len(gemeinschaft))
display(gemeinschaft)

Wohngeb.f.Gemeinschaften: 22


,EGID,BFS_NR,GDEHISTID,BFS_GEMEINDE,ESID,STRNAMK1_HPT,STRNAMK1,DEINR,DEINR_ZUS_HPT,GBEZ,...,GENW2,GENW2_TXT,GWAERSW2,GWAERSW2_TXT,GWAERZW2,GWAERZW2_TXT,GWAERDATW2,KOORD_ORIGIN,DAT_VERARB,geometry
23643,209192,1061.0,15600.0,Luzern,10074455.0,Staffelnhofstrasse,Staffelnhofstr.,60,None,Alterszentrum,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2663632.526 1212739.607)
71725,210167,1061.0,15600.0,Luzern,10074189.0,Rigistrasse,Rigistr.,48,None,Abzubrechen,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2667411.521 1212229.458)
71730,210190,1061.0,15600.0,Luzern,10015131.0,Tivolistrasse,Tivolistr.,21,None,Gemeinschaftszentrum St. Anna,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2667400.924 1212204.956)
24481,210191,1061.0,15600.0,Luzern,10015131.0,Tivolistrasse,Tivolistr.,5,None,Schwestern-Pflegerinnenheim,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2667406.406 1212181.044)
24505,210223,1061.0,15600.0,Luzern,10179092.0,Haldenstrasse,Haldenstr.,41,None,ABZUBRECHEN Personalhaus,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666926.27 1212011.919)
25004,210797,1061.0,15600.0,Luzern,10105676.0,Kapuzinerweg,Kapuzinerweg,12,None,Pflegheim,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666425.252 1212435.036)
25005,210798,1061.0,15600.0,Luzern,10105676.0,Kapuzinerweg,Kapuzinerweg,39,None,Kinderheim,...,NaN,None,NaN,None,NaN,None,NaT,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666403.529 1212498.307)
25033,210828,1061.0,15600.0,Luzern,10108501.0,Wesemlinring,Wesemlinring,7,None,Kinderheim Titlisblick,...,7500.0,Keine,860.0,Gemäss Volkszählung 2000,7600.0,Kein Wärmeerzeuger,2001-11-29 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2666340.69 1212603.367)
25377,211242,1061.0,15600.0,Luzern,10052328.0,Rosenbergstrasse,Rosenbergstr.,2,None,Alterszentrum Rosenberg,...,7570.0,Sonne (thermisch),869.0,Gemäss Baubewilligung,7620.0,Thermische Solaranlage,2016-07-04 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2665829.435 1212605.401)
25378,211243,1061.0,15600.0,Luzern,10052328.0,Rosenbergstrasse,Rosenbergstr.,4,None,Alterszentrum Rosenberg,...,7570.0,Sonne (thermisch),869.0,Gemäss Baubewilligung,7620.0,Thermische Solaranlage,2021-09-27 00:00:00+00:00,Gebäude,2026-08-11 00:00:00+00:00,POINT (2665856.387 1212611.214)
